## Introduction to SPARQL
In this module, we will cover the basics of using SPARQL.

SPARQL (<b>S</b>imple <b>P</b>rotocol <b>A</b>nd <b>R</b>DF <b>Q</b>uery <b>L</b>anguage) is a query language (similar to SQL) used to select and modify data modeled as a graph. 
Conceptually, data modeled as graph consists of a subject, object, and a predicate (or relation) that holds between the the subject and object:

<img width="300px" src="images/subject-predicate-object.png" />

Syntactically, these graphs form statements called "triples". E.g.:

Fido is-a dog.
Fido eats bones.

Technically, these statements often form "quads" that include the name of graph in which the statement resides. The precise syntax for how to form triples/quads is beyond the scope of this module, but the interested reader can refer to:
* https://www.w3.org/TR/rdf-primer/
* https://www.w3.org/RDF/
* https://www.w3.org/TR/rdf-schema/
* https://www.w3.org/TR/owl-ref/

For this module, we will use the Python library `rdflib` to execute our queries:
* https://rdflib.readthedocs.io/en/stable/

Information about SPARQL is found here:
* https://www.w3.org/TR/rdf-sparql-query/
* https://en.wikibooks.org/wiki/SPARQL

In [1]:
from rdflib import Graph

To load an ontology for querying, we call the `parse` method on the file holding the data. For this module, I created an OWL file by extracting statements about cardiocytes from the cell ontology.

"OWL" stands for "Web Ontology Language":
* https://www.w3.org/TR/owl-ref/

I do not know why the creators of OWL decided to switch the "O" and "W" :(

The cell ontology is a structured controlled vocabulary for cell types in animals. Information about it is found here:
* http://www.obofoundry.org/ontology/cl.html

As for what an "ontology" is ... that is a potential long discussion. For now, let's just say that an ontology is a set of formal axioms that describe a domain of interest. In this case, the formal statements are made using OWL, and the domain of interest is cardiocytes.

If you wish to view the OWL file directly, you do this by opening the file in a text editor. However, this would be difficult to interpret. A better option would be to open the file using the Protégé ontology editor:
*

If you are interested in learning about ontologies and how to use Protégé, I have included a tutorial named `Protege4_v1_3_Pizza_Tutorial.pdf` in the same directory as this module. The tutorial was written for an older version of Protégé, but the concepts are still applicable. You are to work through this at your leisure :)

In [2]:
ontology = Graph().parse("ontology/cardiocyte.owl")

## Select query
For our first query, we will find all subclasses of somatic cell. A class, in this sense, is grouping for a set of individuals. For example, the class `human being` contains all the individual humans. In OWL, classes are similar to the classic mathematical notion of set, except that two classes can contain the same individuals while remaining distinct classes. Sets, classically understood, are identifical (i.e., the same set) if each contain the same individuals.

Classes can be grouped into hierarchies. If class `A` is a subclass of class `B`, then any individual contained in class `A` is also contained in class `B`. In such a case, we say that class `A` is a child of class `B`. A hierarchy can be arbitrarily deep, consisting of child classes, parent classes, grandparent classes, etc.

<img width="300px" src="images/class-subclass.png" />

In the cell ontology, classes of cells form hierarchies. For instance, cardiocyte and neural cell are children of somatic cell, which is in turn a child of native cell:

<img width="300px" src="images/cardiocyte-somtatic-cell.png" />

The classes in the cell ontology are uniquely identified using an Internationalized Resource Identifier (IRI):
* https://en.wikipedia.org/wiki/Internationalized_Resource_Identifier

Sometimes you will also see this called an Uniform Resource Identifier (URI). The difference between an IRI and URI is that an IRI permits the use of non-ASCII characters. 

In addition to the IRI, the cell ontology also provides a human readable label for the class. The reason for having IRIs is that allows machines to easily identify a class within an ontology even though it may have the same label as a class in another ontology. For example, the NCI Thesaurus also has class named 'Somatic Cell', but that class has a different IRI.

## Namespaces
When executing SPARQL queries, it is often convenient to make use of namespaces:
* https://www.w3.org/TR/rdf-sparql-query/#docNamespaces

In the query below, we have defined two namespaces: `obo` and `rdfs`.

Doing this allows us to more easily reference entities in the ontology. The namespaces serve as shortand way of writing the full IRI. For instance, the `obo` namespace (or prefix) is defined as `http://purl.obolibrary.org/obo/`. When `obo:` used in a query, the query processor will substitute `http://purl.obolibrary.org/obo/` everywhere is finds `obo:`. For example, `obo:CL_0002371` becomes `http://purl.obolibrary.org/obo/CL_0002371`.

Within the cell ontology, class/sublcass relations are defined using the `subClassOf` relation, and human readable labels are associated with classes using the `label` relation. These relations are defined in the `rdfs` namespace:
* https://www.w3.org/TR/rdf-schema/#ch_subclassof
* https://www.w3.org/TR/rdf-schema/#ch_label

In the RDFS and OWL documentation, what I refer to as "relations" (or predicates) are called "properties". I know ... confusing, but these are relations that hold between entities. Also, the human readable label (i.e., the value of `rdfs:label`) is often simply referred to as an "annotation" because it is metadata that cannot be formally defined in OWL:
* https://www.w3.org/TR/owl2-syntax/#Annotation_Properties

Annotations are also used to provide comments and definitions associates with IRIs. Examples are given below.

In the query below, we define two namespaces (discussed above) and use the `select` keyword to specifiy that we are wanting to return the results that match the pattern specified in the `where` clause of our query (as opposed to deleting or updating the results). The results are then stored in the variables `?iri` and `?label`. Variables in SPARQL begin with a `?`, and the `#` is used to write comments within the query.
```
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

select ?iri ?label 
where {
  ?iri rdfs:subClassOf obo:CL_0002371 . # find all subclasses of somatic cell
  ?iri rdfs:label ?label .
}
```

The `where` clause consists of two patterns:
```
?iri rdfs:subClassOf obo:CL_0002371 .
?iri rdfs:label ?label .
```

The first pattern says (roughly) to match the variable `?iri` to the subject of any statement that asserts the subject is a subclass of `obo:CL_0002371`.

The second pattern says (roughly) to match the variable `?label` to the object of any statement using `rdfs:label` to relate the subject (i.e., the value of `?iri`) to the object.

The firgure above, can be represented by statments (triples):
```
obo:CL_0002371 rdfs:label "somatic cell" .

obo:CL_0002494 rdfs:subClassOf obo:CL_0002371 .
obo:CL_0002494 rdfs:label "cardiocyte" .

obo:CL_0002319 rdfs:subClassOf obo:CL_0002371 .
obo:CL_0002319 rdfs:label "neural cell" .
```
If we were to execute the SPARQL query against these statments, the variable `?iri` would be matched to `obo:CL_0002494` and `obo:CL_0002319`, since these are the subjects of the statments.

The variable `?label` would be matched to `cardiocyte` and `neural cell` because these are objects that have been related to the subject (i.e., `?iri`) using the `rdfs:label` relation.

When patterns are listed in sequence (as they are above), a result is result is returned only if all the variables have been matched. For instance, the query will return classes that are subclasses of somatic cell AND have a label associated with them. SPARQL also allows you to return partial results, but we won't cover this yet.

For reference, here is the view of the somatic cell hierarchy using Protégé:

<img width="400px" src="images/somatic_cell_hierarchy.png" />

The workflow we will use for executing queries in `rdflib` will be to first build the SPARQL inside of a string, and second to store the results of the query in another variable.

In [3]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

select ?iri ?label where {
  ?iri rdfs:subClassOf obo:CL_0002371 . # find all subclasses of somatic cell
  ?iri rdfs:label ?label .
}
"""
results = ontology.query(query)

Executing the query returns a `SPARQLResult` object, which does not permit you to directly view the results.

In [4]:
print(results)

In order to see the results of the query, you need to iterate over the items in `results`.

In [5]:
for r in results:
    print(r)

(rdflib.term.URIRef('http://purl.obolibrary.org/obo/CL_0002319'), rdflib.term.Literal('neural cell', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('http://purl.obolibrary.org/obo/CL_0002077'), rdflib.term.Literal('ecto-epithelial cell', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('http://purl.obolibrary.org/obo/CL_0000349'), rdflib.term.Literal('extraembryonic cell', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('http://purl.obolibrary.org/obo/CL_0000068'), rdflib.term.Literal('duct epithelial cell', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('http://purl.obolibrary.org/obo/CL_0002076'), rdflib.term.Literal('endo-epithelial cell', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('http://purl.obolibrary.org/obo/CL_0002320'), rdflib.term.Literal('connecti

Since the raw `rdflib` results are a bit difficult to interpret, we will instead print out the results using syntax similar to printing out results from a dataframe. `Rdflib` allows you to do this by design.

Below, we print out out IRI and label using the dot notation (e.g., `r.iri`). However, you can also use the bracket notation if needed (e.g., `r['iri']`).

You can verify the results by examing the classes that are one level below the somatic cell class in the Protégé figure above.

In [6]:
for r in results:
    print(r.iri, r.label)


http://purl.obolibrary.org/obo/CL_0002319 neural cell
http://purl.obolibrary.org/obo/CL_0002077 ecto-epithelial cell
http://purl.obolibrary.org/obo/CL_0000349 extraembryonic cell
http://purl.obolibrary.org/obo/CL_0000068 duct epithelial cell
http://purl.obolibrary.org/obo/CL_0002076 endo-epithelial cell
http://purl.obolibrary.org/obo/CL_0002320 connective tissue cell
http://purl.obolibrary.org/obo/CL_0002494 cardiocyte


In addition to labels, the cell ontology includes a number of other important annotations for its classes.

In this query, we are also interested in seeing the definitions (i.e., `?definition` that have been provided for each class). For convenience, we have limited the results using the `limit 3` clause.

In [7]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix definition: <http://purl.obolibrary.org/obo/IAO_0000115>

select ?iri ?label ?definition where {
  ?iri rdfs:subClassOf obo:CL_0002371 . # find all subclasses of somatic cell
  ?iri rdfs:label ?label .
  ?iri definition: ?definition .
}
order by ?label
limit 3
"""
results = ontology.query(query)

We print out the the results using an "f string":
* https://realpython.com/python-f-strings/

This allows us to easily subsitute in the variable values within the string.

In [8]:
for r in results:
    print(f"""
Label: {r.label}
IRI: {r.iri}
Definition: {r.definition}""")


Label: cardiocyte
IRI: http://purl.obolibrary.org/obo/CL_0002494
Definition: A cell located in the heart, including both muscle and non muscle cells.

Label: connective tissue cell
IRI: http://purl.obolibrary.org/obo/CL_0002320
Definition: A cell of the supporting or framework tissue of the body, arising chiefly from the embryonic mesoderm and including adipose tissue, cartilage, and bone.

Label: ecto-epithelial cell
IRI: http://purl.obolibrary.org/obo/CL_0002077
Definition: An epithelial cell derived from ectoderm.


### Filtering results
In practice, it is often necessary to filter the results according to one or more criteria. For this, we use a `filter` statement inside of the `where` clause:
* https://en.wikibooks.org/wiki/SPARQL/FILTER
* https://www.w3.org/TR/sparql11-query/#scopeFilters
* https://jena.apache.org/tutorials/sparql_filters.html

For instance, suppose we want to find classes that are cross referenced with the Foundational Model of Anatomy (FMA) entity idenitified by `FMA:83808`. 
<img width="500px" src="images/cardiocyte-hasDbXref.png" />

In the cell ontology, we would filter our query results using the values of the `hasDbXref` annotation.
```
filter(?xref = "FMA:83808")
```

In [9]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix definition: <http://purl.obolibrary.org/obo/IAO_0000115>
prefix hasDbXref: <http://www.geneontology.org/formats/oboInOwl#hasDbXref>

select ?iri ?label ?definition ?xref where {
  ?iri rdfs:subClassOf obo:CL_0002371 . # find all subclasses of somatic cell
  ?iri rdfs:label ?label . 
  ?iri definition: ?definition .
  ?iri hasDbXref: ?xref .
       
  filter(?xref = "FMA:83808")
}
"""
results = ontology.query(query)

In [10]:
for r in results:
    print(f"""
Label: {r.label}
IRI: {r.iri}
DbXref: {r.xref}
Definition: {r.definition}""")


Label: cardiocyte
IRI: http://purl.obolibrary.org/obo/CL_0002494
DbXref: FMA:83808
Definition: A cell located in the heart, including both muscle and non muscle cells.


## Property paths
The above query returned the number of children of the somatic cell class. But what if we want to also return the children or grandchildren of somatic cell? For instance, the class retinal cell is a grandchild fo somatic cell.

To return all the decendants of somatic cell, we use SPARQL property paths:
* https://www.w3.org/TR/sparql11-property-paths/

A property path is a possible route through a graph between two graph nodes. A trivial case is a property path of length exactly 1, which is a triple pattern. 

Path operators are appended (or prepended) to the predicates of the triples. The `+` operator follows all routes that have **one or more** occurrences of the predicate. The `*` operator follows all routes that have **zero or more** occurrences of the predicate.

So, appending a `+` to the `rdfs:subClassOf` predicate (i.e., `rdfs:subClassOf+`) will instruct the query processor to follow a path of triples that are chained together by the `rdfs:subClassOf` predicate.
<img width="450px" src="images/somatic-cell-property-path-plus.png" />

Now when we run the query, we get a count of all the decendants of somatic cell. You can verify the result using the Protégé image above.

Try appending the `*` operator (i.e., `rdfs:subClassOf*`). Do you know why the count is different?

In [11]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

select (count(?iri) as ?total) where {
  ?iri rdfs:subClassOf+ obo:CL_0002371 . # find all subclasses of somatic cell
  ?iri rdfs:label ?label .
}
"""
results = ontology.query(query)

In [12]:
for r in results:
    print(r.total)

13


You can also specify arbitrary paths. For example, the path:
```
rdfs:subClassOf/rdfs:subClassOf
```
follows the `rdfs:subClassOf` predicates that are two levels deep (i.e., grandchildren).

In [13]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

select ?iri ?label where {
  # find grandchildren of somatic cell and their labels
  ?iri rdfs:subClassOf/rdfs:subClassOf obo:CL_0002371; 
       rdfs:label ?label .
}
"""
results = ontology.query(query)

In [14]:
for r in results:
    print(f"""{r.iri} {r.label}""")

http://purl.obolibrary.org/obo/CL_0009004 retinal cell
http://purl.obolibrary.org/obo/CL_0002159 general ecto-epithelial cell
http://purl.obolibrary.org/obo/CL_0002251 epithelial cell of alimentary canal
http://purl.obolibrary.org/obo/CL_0002503 adventitial cell


## Optional information
As noted above, SPARQL returns results that match all the patterns in the `where` clause. For example, the query below matches all subclasses of somatic cell that have a label and a comment.

This query uses a convenient shorthand notation. When the subject is repeated multiple times, you end the line with a `;` and specify only the predicate and object on the next line. The final line should end with a `.`. In other words, the pattern:
```
?iri rdfs:subClassOf obo:CL_0002371 . # find all subclasses of somatic cell
       rdfs:label ?label;
       rdfs:comment ?comment .
```
is equivalent to:
```
?iri rdfs:subClassOf obo:CL_0002371; # find all subclasses of somatic cell
?iri rdfs:label ?label .
?iri rdfs:comment ?comment .
```

In [15]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix definition: <http://purl.obolibrary.org/obo/IAO_0000115>

select ?iri ?label ?comment where {
  ?iri rdfs:subClassOf obo:CL_0002371; # find all subclasses of somatic cell
       rdfs:label ?label;
       rdfs:comment ?comment .
}
order by ?label
"""
results = ontology.query(query)

In [16]:
for r in results:
    print(f"""
Label: {r.label}
IRI: {r.iri}
Comment: {r.comment}""")


Label: cardiocyte
IRI: http://purl.obolibrary.org/obo/CL_0002494
Comment: From Onard of the FMA: Cardiac muscle cell or cardiac myocyte is a striated muscle cell. Cardiocyte on the other hand is any cell in the heart which includes cells other than muscle cells (e.g. endothelial cell of endocardium). Unless there is a consensus among anatomists that cardiocytes refer only to muscle cells, we will treat them as a general class of cells in the heart.


But how can this result be correct? We now from the above queries that there 7 children classes of somatic cell, and 13 descendants in total.

In fact, the result is correct because the SPARQL return exactly the results that:
* match `rdfs:label ?label`
* AND match `rdfs:comment ?comment`

If you want to include matches that may or may not have a comment, you use the `optional` keyword:
```
optional { ?iri rdfs:comment ?comment }
```
       

In [17]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix definition: <http://purl.obolibrary.org/obo/IAO_0000115>

select ?iri ?label ?comment where {
  ?iri rdfs:subClassOf obo:CL_0002371; # find all subclasses of somatic cell
       rdfs:label ?label .
       
  ## not all iris have comments, so make optional
  optional { 
   ?iri rdfs:comment ?comment 
  } .
}
order by ?label
limit 3
"""
results = ontology.query(query)

In [18]:
for r in results:
    print(f"""
Label: {r.label}
IRI: {r.iri}
Comment: {r.comment}""")


Label: cardiocyte
IRI: http://purl.obolibrary.org/obo/CL_0002494
Comment: From Onard of the FMA: Cardiac muscle cell or cardiac myocyte is a striated muscle cell. Cardiocyte on the other hand is any cell in the heart which includes cells other than muscle cells (e.g. endothelial cell of endocardium). Unless there is a consensus among anatomists that cardiocytes refer only to muscle cells, we will treat them as a general class of cells in the heart.

Label: connective tissue cell
IRI: http://purl.obolibrary.org/obo/CL_0002320
Comment: None

Label: duct epithelial cell
IRI: http://purl.obolibrary.org/obo/CL_0000068
Comment: None


## Aggregate query
SPARQL has limited number of functions that allow you to perform operations over groups:
* https://www.w3.org/TR/sparql11-query/#aggregates

Common aggegrate functions include counting the number of items matched, finidng the minimum or maximum values, and summing up values. Below, we simply find the number (i.e., `count`) of the number of classes that are subclasses of somatic cell.

When performing an aggregate query, you must sure to place the aggregation funcion withing parenthesis and assign a variable to hold the result of the function using the `as` keyword. E.g.: 
```
(count(?iri) as ?total)
```

In [19]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

select (count(?iri) as ?total) where {
  ?iri rdfs:subClassOf obo:CL_0002371 . # find all subclasses of somatic cell
  ?iri rdfs:label ?label .
}
"""
results = ontology.query(query)

In [20]:
for r in results:
    print(r.total)

7


### Aggregate functions with GROUP BY
Above we retrieved a simple count of the subclasses of somatic cell. Now supposed we wanted to know how many comments were associated with each class. For this, we would retrieve each subclass and the count of the class' comments. The `select` clause would look like this:
```
select ?iri ?label (count(?comment) as ?comment_count)
```
But when the data is returned, you have to need to specify how to partition (or group) the data:
* https://en.wikibooks.org/wiki/SPARQL/Aggregate_functions
* https://jena.apache.org/documentation/query/group-by.html

To do this, you use the `group by` function after the `where` clause.

In [21]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

select ?iri ?label (count(?comment) as ?comment_count)
where {
  ?iri rdfs:subClassOf+ obo:CL_0002371; # find all subclasses of somatic cell
       rdfs:label ?label .
  
  ## optionally find comments for each class
  optional {
    ?iri rdfs:comment ?comment 
  } .
}
group by ?iri ?label
"""
results = ontology.query(query)

In [22]:
for r in results:
    print(f"""
Label: {r.label}
IRI: {r.iri}
Comment Count: {r.comment_count}""")


Label: neural cell
IRI: http://purl.obolibrary.org/obo/CL_0002319
Comment Count: 0

Label: retinal cell
IRI: http://purl.obolibrary.org/obo/CL_0009004
Comment Count: 0

Label: ecto-epithelial cell
IRI: http://purl.obolibrary.org/obo/CL_0002077
Comment Count: 0

Label: general ecto-epithelial cell
IRI: http://purl.obolibrary.org/obo/CL_0002159
Comment Count: 0

Label: epidermal cell
IRI: http://purl.obolibrary.org/obo/CL_0000362
Comment Count: 0

Label: epithelial cell of skin gland
IRI: http://purl.obolibrary.org/obo/CL_0002308
Comment Count: 0

Label: extraembryonic cell
IRI: http://purl.obolibrary.org/obo/CL_0000349
Comment Count: 0

Label: duct epithelial cell
IRI: http://purl.obolibrary.org/obo/CL_0000068
Comment Count: 0

Label: endo-epithelial cell
IRI: http://purl.obolibrary.org/obo/CL_0002076
Comment Count: 0

Label: epithelial cell of alimentary canal
IRI: http://purl.obolibrary.org/obo/CL_0002251
Comment Count: 0

Label: connective tissue cell
IRI: http://purl.obolibrary.org

### Aggregate functions with HAVING
At times, we may wish to filter the values of our aggregate values. To do, you use the `having` keyword followed by filtering criteria. For instance, the following limits the results to only those in which the comment count is greater than `0`.

In [23]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

select ?iri ?label (count(?comment) as ?comment_count)
where {
  ?iri rdfs:subClassOf+ obo:CL_0002371; # find all subclasses of somatic cell
       rdfs:label ?label .
  
  ## optionally find comments for each class
  optional {
    ?iri rdfs:comment ?comment 
  } .
}
group by ?iri ?label
having (count(?comment) > 0)
"""
results = ontology.query(query)

In [24]:
for r in results:
    print(f"""
Label: {r.label}
IRI: {r.iri}
Comment Count: {r.comment_count}""")


Label: cardiocyte
IRI: http://purl.obolibrary.org/obo/CL_0002494
Comment Count: 1


## Restrictions
An interesting aspect of ontologies built in OWL is that you specify logical relations between  classes. For example, some entities have been defined as being part of the heart.

<img width="500px" src="images/cardiac-chamber-part-of-heart.png" />

When relations are defined between in classes in OWL, a quantifier (e.g., `some` or `only`) is required. The reason for this is that classes range over multiple individuals. 

To see this, let's suppose that `x` is an integer. Now consider the difference between these two statements:
```
x is greater than 3
some x is greater than 3
```

The truth of first statement depends on the particular value of `x`. But, the second statement is always true. When we model a domain using an ontology, we want to only assert axioms that are true. Hence, we relate cardiatic chamber to heart with the axiom `part of some heart` instead of `part of heart`. Every cardiac chamber is part of **some** heart, but a particular cardiac chamber may not be part of a particular heart. For instance, the cardiac chamber in my heart is not part of your heart.

To query axioms (such as `part of some heart`), we make use of restrictions (or property restrictions):
* https://www.w3.org/TR/owl2-syntax/#Object_Property_Restrictions
* https://www.w3.org/TR/owl2-primer/#Property_Restrictions
* https://www.cs.vu.nl/~guus/public/owl-restrictions/

The syntax for using restrictions consists of three parts:
* Define a variable to be of type `owl:Restriction`; e.g.: `?restriction rdf:type owl:Restriction`
* specify the property (i.e., relation) that you are querying on. This is done using the `owl:onProperty` predicate; e.g.: `owl:onProperty part_of:`
* specify the quantifier (e.g. `some`) and the class that the quantifier ranges over. Two common quantifiers are `owl:someValuesFrom` (for `some`) and `owl:allValuesFrom` for (`only`); e.g. `owl:someValuesFrom heart:`.

Putting these peices together, a SPARQL restriciton looks like this:
```
prefix part_of: <http://purl.obolibrary.org/obo/BFO_0000050>
prefix heart: <http://purl.obolibrary.org/obo/UBERON_0000948>

?restriction rdf:type owl:Restriction;   # define varialbe as type of restriction
             owl:onProperty part_of:;    # specify the relation/property
             owl:someValuesFrom heart: . # specify quantifier and target of quantifier
```
This only gets us part of the information we want. We still need to find the classes that have been defined using the restriction. This is done using the `rdfs:subClassOf` or `owl:equivalentClass` predicate; e.g.:
```
?iri rdfs:subClassOf ?restriction
```
Conceptually, this is telling the query processor to find classes that are subclasses of the `the class of things that are part of some heart`. The parent class in this case (i.e., `the class of things that are part of some heart`), however, has not been given name. Such classes are called anonymous classes. Classes that have been given names (e.g., `heart`, `somatic cell`) are called (you guessed it) named classes.

In [25]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix owl: <http://www.w3.org/2002/07/owl#>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix definition: <http://purl.obolibrary.org/obo/IAO_0000115>
prefix part_of: <http://purl.obolibrary.org/obo/BFO_0000050>
prefix heart: <http://purl.obolibrary.org/obo/UBERON_0000948>

select ?iri ?label ?definition where {
  # define restriction for 'part of' some 'heart'
  ?restriction rdf:type owl:Restriction;
               owl:onProperty part_of:;
               owl:someValuesFrom heart: .
               
  ?iri rdfs:subClassOf ?restriction; 
       rdfs:label ?label;
       definition: ?definition .
}
order by ?label
"""
results = ontology.query(query)

In [26]:
for r in results:
    print(f"""
Label: {r.label}
IRI: {r.iri}
Definition: {r.definition}""")


Label: cardiac chamber
IRI: http://purl.obolibrary.org/obo/UBERON_0004151
Definition: A cardiac chamber surrounds an enclosed cavity within the heart

Label: cardiac mesenchyme
IRI: http://purl.obolibrary.org/obo/UBERON_0009751
Definition: The embryonic connective tissue made up of loosely aggregated mesenchymal cells, supported by interlaminar jelly, that gives rise to the developing cardiac structures

Label: cardiac septum
IRI: http://purl.obolibrary.org/obo/UBERON_0002099
Definition: The thin membranous structure between the two heart atria or the thick muscular structure between the two heart ventricles.

Label: coronary vessel
IRI: http://purl.obolibrary.org/obo/UBERON_0005985
Definition: Any of the arteries or veins that supply blood to the heart or return blood from the heart muscles to the circulation

Label: heart elastic tissue
IRI: http://purl.obolibrary.org/obo/UBERON_0003610
Definition: The type of heart connective tissue found in the endocardial layer that consists main

Above we found all classes that were subclasses of the anonymous class `the class of things that are part of some heart`. The this example the axiom `part of some heart` defined a property all the members of this class must have. These properties are often referred to as necessarily conditions. 

Necessary conditions, however, do not fully define the members of the anonymous class. For instance, while is true that a `cardiac chamber` is part of the heart, it is also a `mesoderm-derived structure` (i.e., it develops from the mesoderm). The `part of some heart` axiom does not capture this truth.

Sometimes, though, it is possible to provide axioms to do capture all the truths need to define an entity. For example, in the cell ontology, the class `venous blood` is equivalent to `blood and (part of some vein)`.

<img width="500px" src="images/venous-blood-equivalent-class.png" />

This entails two things:
1. Any entity that is a member of the named class `blood` **and** a member of the anonymous class `part of some vein` must also be a member of the named class `venous blood`.
2. Any entity that is a member of the named class `venous blood` must also be a member of the named class `blood` **and** a member of the anonymous class `part of some vein`.

To query for the conjunction of two classes (e.g., `blood and (part of some vein)`) we use the `owl:intersectionOf` predicate:
* https://www.w3.org/TR/owl2-syntax/#Intersection_of_Class_Expressions

And instead of querying for subclasses (i.e., `rdfs:subClassOf`), we query using the predicate `owl:equivalentClass`. Classes that have been defined using `owl:equivalentClass` are often referrred as defined classes.

In [27]:
query = """
prefix obo: <http://purl.obolibrary.org/obo/>
prefix owl: <http://www.w3.org/2002/07/owl#>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix definition: <http://purl.obolibrary.org/obo/IAO_0000115>
prefix part_of: <http://purl.obolibrary.org/obo/BFO_0000050>
prefix blood: <http://purl.obolibrary.org/obo/UBERON_0000178>
prefix venous_blood: <http://purl.obolibrary.org/obo/UBERON_0013756>
prefix vein: <http://purl.obolibrary.org/obo/UBERON_0001638>

select ?iri ?label ?definition where {
  # define restriction for 'part of' some 'vein'
  ?part_of_vein rdf:type owl:Restriction;
                owl:onProperty part_of:;
                owl:someValuesFrom vein: .
  
  # define intersection (or conjunction) of 'blood' and ('part of' some 'vein')
  ?blood_and_vein owl:intersectionOf (blood: ?part_of_vein) .
  
  # find the iri that is equivalent to 'blood' and ('part of' some 'vein')
  ?iri owl:equivalentClass ?blood_and_vein;
       rdfs:label ?label;
       definition: ?definition .
}
"""
results = ontology.query(query)

In [28]:
for r in results:
    print(f"""
Label: {r.label}
IRI: {r.iri}
Definition: {r.definition}""")


Label: venous blood
IRI: http://purl.obolibrary.org/obo/UBERON_0013756
Definition: A blood that is part of a vein.


## Wrapping up
In this module, we have covered a number of important concepts needed to develop a firm understanding of SPARQL:
* Select and filter queries
* Property paths
* Aggregate queries
* Using restrictions

There is still much learn about SPARQL and ontologies, but hopefully this gets you started.